# Run `cfd73968aad1`

**Mode**: `research`
**Tier**: `standard`
**Datasets**: `tasic2018-v1`

## Request

> [bench:research-easy-tasic-celltype-counts] Count cells per cell type (use the 'cluster' field) in the Tasic 2018 V1 dataset. Produce: /work/celltype_counts.csv (two columns: cell_type, n_cells, sorted descending), and /work/celltype_counts_bar.png (horizontal bar plot of the top 30 types). Use the V1-only object pointed at by the dataset's access snippet.

## Final summary

Both artifacts are saved and verified. Here's the wrap-up:

---

## Summary
Loaded the Tasic 2018 V1-only processed object (`v1_all_proc.h5ad`, 13,586 cells) and counted cells by `cell_cluster`. There are **118 distinct cell types**; the top type is *L4 IT VISp Rspo1* (1,350 cells, ~10%). The full ranked table is in the CSV and the top-30 are visualized in the bar chart, color-coded by broad class (excitatory = blue, inhibitory = orange, non-neuronal = gray).

## What I did
- Loaded `/data/tasic2018-v1/v1_all_proc.h5ad` (the dataset card's recommended V1 entry point, 13,586 × 27,505)
- Counted `cell_cluster` values, sorted descending → **`/work/celltype_counts.csv`** (118 rows × 2 columns: `cell_type`, `n_cells`)
- Generated a horizontal bar chart of the top 30 types, colored by class (excitatory/inhibitory/non-neuronal) using the Okabe-Ito colorblind-safe palette → **`/work/celltype_counts_bar.png`**
- Figure follows Tufte data-ink principles: no top/right spines, horizontal grid only, count labels at bar ends, constrained layout

## Caveats / next steps
- The V1 object has **13,586 cells across 118 types** (vs. 14,662 raw VISp cells in the full h5ad) — the processed object applies QC filters, so ~1,076 cells are excluded; this is expected
- Color assignment is rule-based (keyword matching on cluster name), so rare edge cases could be mis-colored — spot-check if using for publication
- The CSV covers all 118 types; the PNG shows only top 30 — if you need all 118 plotted, ask and I can produce a taller figure

In [ ]:
import os
os.environ["NUMBA_DISABLE_JIT"] = "1"
import pandas as pd
import anndata as ad

# Load the full h5ad and filter to VISp only
adata = ad.read_h5ad("/data/tasic2018-v1/tasic2018_full_counts.h5ad")
print("Full shape:", adata.shape)
print("Obs columns:", list(adata.obs.columns))
print("dissected_region values:", adata.obs["dissected_region"].value_counts().to_dict())


exit_code: 0
--- stdout ---
Full shape: (23178, 45768)
Obs columns: ['title', 'source_name', 'organism', 'donor_id', 'donor_sex', 'donor_genotype', 'injection_type', 'injection_target', 'injected_material', 'dissected_region', 'dissected_layer', 'facs_gating', 'facs_date', 'rna_amplification_set', 'sequencing_tube', 'sequencing_batch', 'sequencing_qc_pass_fail', 'cell_class', 'cell_subclass', 'cell_cluster', 'molecule', 'SRA_Run', 'GEO_Sample', 'GEO_Sample_Title']
dissected_region values: {'VISp': 14662, 'ALM': 8516}

--- stderr ---



In [ ]:

# Load the V1-only processed object (as specified by dataset card access snippet)
adata_v1 = ad.read_h5ad("/data/tasic2018-v1/v1_all_proc.h5ad")
print("V1 shape:", adata_v1.shape)
print("Obs columns:", list(adata_v1.obs.columns))
# Preview cluster field
if "cell_cluster" in adata_v1.obs.columns:
    print("\ncell_cluster sample:", adata_v1.obs["cell_cluster"].value_counts().head(5))


exit_code: 0
--- stdout ---
V1 shape: (13586, 27505)
Obs columns: ['title', 'source_name', 'organism', 'donor_id', 'donor_sex', 'donor_genotype', 'injection_type', 'injection_target', 'injected_material', 'dissected_region', 'dissected_layer', 'facs_gating', 'facs_date', 'rna_amplification_set', 'sequencing_tube', 'sequencing_batch', 'sequencing_qc_pass_fail', 'cell_class', 'cell_subclass', 'cell_cluster', 'molecule', 'SRA_Run', 'GEO_Sample', 'GEO_Sample_Title', 'leiden']

cell_cluster sample: cell_cluster
L4 IT VISp Rspo1           1350
L6 IT VISp Penk Col27a1    1031
L2/3 IT VISp Agmat          658
Lamp5 Lsp1                  425
L5 IT VISp Batf3            390
Name: count, dtype: int64

--- stderr ---



In [ ]:

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

# ── 1. Count cells per cluster (all types, not just top-30) ──────────────────
counts = (
    adata_v1.obs["cell_cluster"]
    .value_counts()
    .rename_axis("cell_type")
    .reset_index(name="n_cells")
    .sort_values("n_cells", ascending=False)
    .reset_index(drop=True)
)
print(f"Total clusters: {len(counts)}, total cells: {counts['n_cells'].sum()}")
print(counts.head(10).to_string(index=False))

# ── 2. Save full CSV ─────────────────────────────────────────────────────────
counts.to_csv("/work/celltype_counts.csv", index=False)
print("\nSaved /work/celltype_counts.csv")


exit_code: 0
--- stdout ---
Total clusters: 118, total cells: 13586
                 cell_type  n_cells
          L4 IT VISp Rspo1     1350
   L6 IT VISp Penk Col27a1     1031
        L2/3 IT VISp Agmat      658
                Lamp5 Lsp1      425
          L5 IT VISp Batf3      390
                Pvalb Tpbg      386
L6 IT VISp Col23a1 Adamts2      371
                Astro Aqp4      361
           Pvalb Reln Tac1      328
      L6 CT VISp Ctxn3 Sla      315

Saved /work/celltype_counts.csv

--- stderr ---
mkdir -p failed for path /home/sandbox/.config/matplotlib: [Errno 13] Permission denied: '/home/sandbox/.config/matplotlib'
Matplotlib created a temporary cache directory at /tmp/matplotlib-71d250nr because there was an issue with the default path (/home/sandbox/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.



In [ ]:

# ── 3. Build the top-30 horizontal bar chart ─────────────────────────────────

top30 = counts.head(30).copy()

# Okabe-Ito-inspired palette: color by broad class (excitatory, inhibitory, non-neuronal)
def assign_color(name):
    name_lower = name.lower()
    # Non-neuronal
    if any(k in name_lower for k in ["astro", "endo", "oligo", "micro", "opc", "smc", "vlmc", "peri"]):
        return "#999999"  # gray
    # Inhibitory (GABAergic)
    if any(k in name_lower for k in ["lamp5", "pvalb", "sst", "vip", "sncg", "meis2", "chodl"]):
        return "#E69F00"  # Okabe-Ito orange
    # Excitatory (Glutamatergic) — default
    return "#56B4E9"  # Okabe-Ito sky-blue

colors = [assign_color(ct) for ct in top30["cell_type"]]

# Figure height scaled by number of bars
n = len(top30)
fig_h = 0.28 * n + 0.8
fig, ax = plt.subplots(figsize=(7.5, fig_h), constrained_layout=True)

# Horizontal bars — sorted descending means largest at top
y = np.arange(n)
bars = ax.barh(y, top30["n_cells"], color=colors, height=0.72, edgecolor="none")

# Count labels at bar ends
for bar_obj, val in zip(bars, top30["n_cells"]):
    ax.text(
        bar_obj.get_width() + 10, bar_obj.get_y() + bar_obj.get_height() / 2,
        f"{val:,}", va="center", ha="left", fontsize=7.5, color="#444444"
    )

# Y-axis tick labels (cell type names)
ax.set_yticks(y)
ax.set_yticklabels(top30["cell_type"], fontsize=8.2)
ax.invert_yaxis()  # largest at top

# Axes decoration — Tufte style
ax.set_xlabel("Number of cells", fontsize=10)
ax.set_title(
    f"Tasic 2018 — VISp cell-type composition (top 30 of {len(counts)} types, n={counts['n_cells'].sum():,})",
    fontsize=10, fontweight="semibold", loc="left", pad=8
)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_color("#222222")
ax.spines["bottom"].set_linewidth(0.8)
ax.tick_params(axis="y", length=0)
ax.tick_params(axis="x", labelsize=8.5, color="#222222")

# Subtle vertical grid lines only
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.grid(axis="x", color="#dddddd", linewidth=0.4, zorder=0)
ax.set_axisbelow(True)

# Legend for class color coding
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#56B4E9", label="Excitatory (Glutamatergic)"),
    Patch(facecolor="#E69F00", label="Inhibitory (GABAergic)"),
    Patch(facecolor="#999999", label="Non-neuronal"),
]
ax.legend(
    handles=legend_elements, fontsize=7.5, frameon=False,
    loc="lower right", bbox_to_anchor=(1.0, 0.0)
)

# Extend x-axis so count labels don't clip
ax.set_xlim(0, top30["n_cells"].max() * 1.18)

fig.savefig("/work/celltype_counts_bar.png", dpi=150, bbox_inches="tight", pad_inches=0.15)
plt.close(fig)
print("Saved /work/celltype_counts_bar.png")

# Quick sanity check on CSV
check = pd.read_csv("/work/celltype_counts.csv")
print(f"CSV shape: {check.shape}, columns: {list(check.columns)}")
print(check.head(3).to_string(index=False))


exit_code: 0
--- stdout ---
Saved /work/celltype_counts_bar.png
CSV shape: (118, 2), columns: ['cell_type', 'n_cells']
              cell_type  n_cells
       L4 IT VISp Rspo1     1350
L6 IT VISp Penk Col27a1     1031
     L2/3 IT VISp Agmat      658

--- stderr ---



## Artifacts
- [`celltype_counts.csv`](./celltype_counts.csv)
- [`celltype_counts_bar.png`](./celltype_counts_bar.png)